# 10 — Profiling, Memory, and Benchmarks

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/10', {1: 'a5b72d5e04557ce6', 2: 'c715e3789da6391f', 3: '4fde6271fb9740ea', 4: '367e8ffc914f84cb', 5: '0b44d70915a6eae2', 6: '2fe5008566d89fdb'}, independent='peak_delta')

## 1. Measure before optimising

- Big-O describes growth; real speed also depends on constants, caching and data size. Measure.
- `timeit.repeat(stmt, number=n, repeat=r)` gives several timings; report the minimum (least noise).
- Benchmark realistic inputs and compare like with like, after checking both versions give the same answer.

In [ ]:
import timeit
setup = "data = list(range(2000))"
loop = min(timeit.repeat("[x * 2 for x in data]", setup=setup, number=200, repeat=3))
mapped = min(timeit.repeat("list(map(lambda x: x * 2, data))", setup=setup, number=200, repeat=3))
print(round(loop, 4), round(mapped, 4))

**Exercise 1.** Which number from several `timeit.repeat` runs is the fairest to report: `"min"`, `"max"` or `"mean"`?

In [ ]:
answer = ___
check(1, answer)

**Exercise 2.** Check both versions agree before timing them.

In [ ]:
data = range(10)
a = [x * 2 for x in data]
b = list(map(lambda x: x * 2, data))
same = a == ___
check(2, same)

## 2. Profiling

- `cProfile` counts calls and time per function; sort by `cumulative` to find the hotspot.
- `pstats.Stats(profile).sort_stats("cumulative").print_stats(5)` shows the top five.
- Optimise the function that dominates the time, then measure again.

In [ ]:
import cProfile, pstats, io

def slow():
    return sum(i * i for i in range(20000))

profile = cProfile.Profile()
profile.enable()
for _ in range(5):
    slow()
profile.disable()
out = io.StringIO()
pstats.Stats(profile, stream=out).sort_stats("cumulative").print_stats(3)
print(out.getvalue()[:400])

**Exercise 3.** Count how many times `slow` was called, from the profile stats.

In [ ]:
import cProfile, pstats

def slow():
    return sum(range(100))

profile = cProfile.Profile()
profile.enable()
for _ in range(4):
    slow()
profile.disable()
stats = pstats.Stats(profile).stats
calls = [v[1] for k, v in stats.items() if k[2] == "slow"][0]
predicted = ___
check(3, predicted)

**Exercise 4.** A benchmark reports bytes already live at startup as new allocations. Report the largest nonnegative rise above the first sample, or zero for no observations. Read the stream once. Repair `peak_growth` and write `regression(candidate)`. Your test must pass for the repair and raise AssertionError for the original defect.

Earlier practice, for discussion:

Which sort key shows time including the functions it calls? Write it.

```python
answer = ___
```

In [ ]:
def peak_growth(samples):
    return max(samples, default=0)

def regression(candidate):
    raise NotImplementedError


def observe(repair):
    def original(samples):
        return max(samples, default=0)

    regression(repair)
    try:
        regression(original)
    except AssertionError:
        detected = True
    else:
        detected = False
    inputs = [([400, 410, 398],), ([3, 1],), ([],), ([5],)]
    return [repair(*args) for args in inputs], detected

check(4, peak_growth, observe=observe)

## 3. Memory

- `sys.getsizeof(x)` is the shallow size of one object, not what it contains.
- `tracemalloc.start()` and `get_traced_memory()` show current and peak allocations.
- Generators use constant memory; lists hold every item. `__slots__` shrinks many small objects.

In [ ]:
import sys, tracemalloc
print(sys.getsizeof(list(range(1000))) > sys.getsizeof(x for x in range(1000)))
tracemalloc.start()
data = [str(i) for i in range(10000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(peak > 0)

**Exercise 5.** Which is smaller: a list of a million numbers or a generator over them? Write `"list"` or `"generator"`.

In [ ]:
answer = ___
check(5, answer)

**Exercise 6.** Does `sys.getsizeof` include the items inside a list? Write `True` or `False`.

In [ ]:
answer = ___
check(6, answer)

## Independent build

A memory probe samples live bytes after each batch. Implement peak_delta(samples), the largest increase above the first sample, or 0 for no samples. Accept a single-pass iterable and use constant extra space. Explain one peak this sampling method could miss.

Use tracemalloc to collect samples while consuming the same stream once incrementally and once into a list. Check the two consumers process the same number of items. Report the observed deltas and explain why the allocator and sample timing affect the measurement. Grade the explanation and experiment separately from the function check.

In [ ]:
# Write your implementation here.

In [ ]:
def acceptance_cases(submission):
    assert submission(iter([9, 6, 13, 8])) == 4, 'Check the result for peak_delta(iter([9, 6, 13, 8]))'
    assert submission(iter([9, 3])) == 0, 'Check the result for peak_delta(iter([9, 3]))'
    assert submission(iter([])) == 0, 'Check the result for peak_delta(iter([]))'

check.assess('peak_delta', acceptance_cases)

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** timeit repeats measurements to…

   - **a)** Profile functions
   - **b)** Reduce noise
   - **c)** Use more memory

**2.** cProfile's cumulative time includes…

   - **a)** Time spent in functions it calls too
   - **b)** Only the function's own code
   - **c)** Import time only

**3.** tracemalloc measures…

   - **a)** Disk usage
   - **b)** CPU time
   - **c)** Python memory allocations

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/10', {1: '8f1194861272', 2: 'd5ed86c195fc', 3: '8392db161c11'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [11 — Algorithms, Complexity, and Cache Design](11_algorithms_complexity_caching.ipynb)